<a href="https://colab.research.google.com/github/pedroManuelP/C-digos-em-Python/blob/main/Lista2_do_Projeto_de_Sistemas_de_Controle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

def coefAmort(Mp_percent):
    n = np.log(Mp_percent/100)
    d = np.sqrt(np.power(np.pi,2) + np.power(np.log(Mp_percent/100),2))
    return n/d

def freqNatural(coefAmort,tss, tss_percent):
  if(tss_percent == 5):
    wn = 3/(tss*coefAmort)
  elif(tss_percent == 2):
    wn = wn = 4/(tss*coefAmort)
  return wn

def anguloThetaPhi(si, p):
  if(si.real > p.real):
    angulo = np.atan2((si.imag - p.imag), (p.real - si.real))
    angulo = 180 - np.rad2deg(angulo)
  elif(si.real < p.real):
    angulo = np.atan2((si.imag - p.imag), (si.real - p.real))
    angulo = np.rad2deg(angulo)
  else:
    angulo = 90
  return angulo

In [ ]:
print("Digite o ganho de G(s): "); kg = (int)(input())

i = 0; arrNum = []; x = 0
while(x != 'q'):
  print("Digite o coeficiente de grau " + str(i) + " do numerador: "); i += 1
  x = input()
  if(x != 'q'):
    arrNum = np.append(arrNum, (float)(x))

i = 0; arrDen = []; x = 0
while(x != 'q'):
  print("Digite o coeficiente de grau " + str(i) + " do denominador: "); i += 1
  x = input()
  if(x != 'q'):
    arrDen = np.append(arrDen, (float)(x))

g_num = kg*np.polynomial.Polynomial(arrNum); g_den = np.polynomial.Polynomial(arrDen)
print("G(s) = (" + str(g_num) + ")/"); print("(" + str(g_den) + ")")

kh = 0.4; h_num = kh*np.polynomial.Polynomial([1]); h_den = np.polynomial.Polynomial([1])
print("\nH(s) = (" + str(h_num) + ")/"); print("(" + str(h_den) + ")")

Digite o ganho de G(s): 
5
Digite o coeficiente de grau 0 do numerador: 
1
Digite o coeficiente de grau 1 do numerador: 
q
Digite o coeficiente de grau 0 do denominador: 
20
Digite o coeficiente de grau 1 do denominador: 
22
Digite o coeficiente de grau 2 do denominador: 
12
Digite o coeficiente de grau 3 do denominador: 
1
Digite o coeficiente de grau 4 do denominador: 
q
G(s) = (5.0)/
(20.0 + 22.0·x + 12.0·x² + 1.0·x³)

H(s) = (0.4)/
(1.0)


In [ ]:
Mp_percent = 20; tss = 5; tss_percent = 2

ksi = np.abs(coefAmort(Mp_percent)); ksi = np.round(ksi,4)
wn = freqNatural(ksi,tss,tss_percent); wn = np.round(wn,4)
print("Coeficiente de amortecimento: " + str(ksi))
print("Frequência natural: " + str(wn))

Coeficiente de amortecimento: 0.4559
Frequência natural: 1.7548


In [ ]:
wd = wn*np.sqrt(1-np.power(ksi,2)); wd = np.round(wd,4)
print("Frequência amortecida: " + str(wd))

s1 = np.round(complex(-ksi*wn,wd), 4); s2 = np.round(complex(-ksi*wn,-wd), 4)

Frequência amortecida: 1.5618


In [ ]:
print("s1: " + str(s1))
print("s2: " + str(s2))

s1: (-0.8+1.5618j)
s2: (-0.8-1.5618j)


In [ ]:
# Controlador PD

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
#print(polos); print(zeros)

somatorioTheta = 0; somatorioPhi = 0
for i in range(len(polos)):
  somatorioTheta += anguloThetaPhi(s1, polos[i])
for i in range(len(zeros)):
  somatorioPhi += anguloThetaPhi(s1, zeros[i])
phiZ = somatorioTheta - somatorioPhi - 180
#z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))                          # para um zero a esquerda de si
z = -s1.real - s1.imag/np.tan(np.deg2rad(180 - phiZ))                   # para um zero a direita de si
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

produtorioA = 1; produtorioB = 1
for i in range(len(polos)):
  produtorioA *= abs(s1 - (polos[i]))
for i in range(len(zeros)):
  produtorioB *= abs(s1 - (zeros[i]))
kt = produtorioA/produtorioB
kc = kt/(kg*kh)
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kd = kc; kp = kd*z
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho derivativo do controlador: " + str(round(kd, 4)))

Zero(s) do controlador projetado: 1.3389
Ganho do controlador projetado: 11492.984

Ganho proporcional do controlador: 15387.7988
Ganho derivativo do controlador: 11492.984


In [ ]:
# Controlador PI

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
polos = np.append(polos, 0); #print(polos); print(zeros)

somatorioTheta = 0; somatorioPhi = 0
for i in range(len(polos)):
  somatorioTheta += anguloThetaPhi(s1, polos[i])
for i in range(len(zeros)):
  somatorioPhi += anguloThetaPhi(s1, zeros[i])
phiZ = somatorioTheta - somatorioPhi - 180
#z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))                          # para um zero a esquerda de si
z = -s1.real - s1.imag/np.tan(np.deg2rad(180 - phiZ))                   # para um zero a direita de si
zeros = np.append(zeros, -z); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

produtorioA = 1; produtorioB = 1
for i in range(len(polos)):
  produtorioA *= abs(s1 - (polos[i]))
for i in range(len(zeros)):
  produtorioB *= abs(s1 - (zeros[i]))
kt = produtorioA/produtorioB
kc = kt/(kg*kh)
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kp = kc; ki = kp*z
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho integrativo do controlador: " + str(round(ki, 4)))

Zero(s) do controlador projetado: 3.4286
Ganho do controlador projetado: 7.0

Ganho proporcional do controlador: 7.0
Ganho integrativo do controlador: 24.0


In [ ]:
# Controlador PID

polos = (g_den*h_den).roots(); zeros = (g_num*h_num).roots()  # polos e zeros da FT de malha fechada do sistema
polos = np.append(polos, 0); #print(polos); print(zeros)

somatorioTheta = 0; somatorioPhi = 0
for i in range(len(polos)):
  somatorioTheta += anguloThetaPhi(s1, polos[i])
for i in range(len(zeros)):
  somatorioPhi += anguloThetaPhi(s1, zeros[i])
phiZ = (somatorioTheta - somatorioPhi - 180)/2
z = -s1.real + s1.imag/np.tan(np.deg2rad(phiZ))                          # para um zero a esquerda de si
#z = -s1.real - s1.imag/np.tan(np.deg2rad(180 - phiZ))                   # para um zero a direita de si
zeros = np.append(zeros, [-z, -z]); print("Zero(s) do controlador projetado: " + str(round(z, 4)))

produtorioA = 1; produtorioB = 1
for i in range(len(polos)):
  produtorioA *= abs(s1 - (polos[i]))
for i in range(len(zeros)):
  produtorioB *= abs(s1 - (zeros[i]))
kt = produtorioA/produtorioB
kc = kt/(kg*kh)
print("Ganho do controlador projetado: " + str(round(kc, 4)))

kp = 0; ki = 0; kd = 0
print("\nGanho proporcional do controlador: " + str(round(kp, 4)))
print("Ganho integrativo do controlador: " + str(round(ki, 4)))
print("Ganho derivativo do controlador: " + str(round(kd, 4)))

Zero(s) do controlador projetado: 2.049
Ganho do controlador projetado: 3.137

Ganho proporcional do controlador: 0
Ganho integrativo do controlador: 0
Ganho derivativo do controlador: 0
